EBS (Elastic Block Store) is the primary block storage service for EC2 — durable, network-attached volumes that persist independently of the instance lifecycle. EC2 Instance Store is the alternative: physically attached NVMe storage that is blazingly fast but ephemeral. Knowing when to use each, and how to manage EBS volumes effectively, is a core SAA-C03 topic.

## EBS Fundamentals

An EBS volume is a **network-attached block storage device** — think of it as a USB drive that AWS plugs into your EC2 instance over the network. Because it's network-attached:

- It can be **detached from one instance and reattached to another** (in the same AZ)
- It **persists after instance termination** (unless Delete on Termination is checked)
- There is a small network latency compared to physically attached storage

### AZ Scope
EBS volumes are **AZ-scoped**. A volume in `us-east-1a` can only be attached to an EC2 instance also in `us-east-1a`. To move a volume to a different AZ or region, you must snapshot it first.

### Delete on Termination
Each volume attachment has a **Delete on Termination** flag:
- **Root volume**: enabled by default (deleted when instance terminates)
- **Additional volumes**: disabled by default (survive instance termination)

You can change this at launch time or while the instance is running.

### Multi-Attach
Most EBS volumes can only be attached to **one instance at a time**. The exception is `io1`/`io2` volumes, which support **Multi-Attach** — attaching to up to 16 instances in the same AZ simultaneously. Multi-Attach requires a cluster-aware filesystem (e.g., GFS2) — standard Linux filesystems like ext4 or XFS will corrupt data if written from multiple instances concurrently.

## EBS Volume Types

| Type | Category | Max IOPS | Max Throughput | Use case |
|---|---|---|---|---|
| **gp3** | General Purpose SSD | 16,000 | 1,000 MB/s | Default for most workloads |
| **gp2** | General Purpose SSD | 16,000 | 250 MB/s | Legacy default (gp3 preferred) |
| **io2 Block Express** | Provisioned IOPS SSD | 256,000 | 4,000 MB/s | Largest databases, highest perf |
| **io1** / **io2** | Provisioned IOPS SSD | 64,000 | 1,000 MB/s | I/O-intensive databases |
| **st1** | Throughput Optimized HDD | 500 | 500 MB/s | Big data, log processing, sequential |
| **sc1** | Cold HDD | 250 | 250 MB/s | Infrequent access, lowest cost |

### gp3 vs gp2

**gp3** is the current default and is almost always preferred:

| | gp2 | gp3 |
|---|---|---|
| Baseline IOPS | 3 IOPS/GB (min 100, max 16,000) | **3,000 IOPS always** |
| IOPS provisioning | Tied to volume size | **Independent of size** |
| Throughput | Up to 250 MB/s | Up to **1,000 MB/s** |
| Cost | Baseline | ~20% cheaper than gp2 |

With gp2, to get 6,000 IOPS you need a 2 TB volume. With gp3, you get 3,000 IOPS on any size volume and can provision more IOPS independently.

### io1 / io2 (Provisioned IOPS)

Use these when:
- You need more than 16,000 IOPS
- You need sustained I/O performance for critical databases (Oracle, SQL Server, SAP HANA)
- You need Multi-Attach

**io2 vs io1**: io2 offers 100x more durability (99.999% vs 99.8–99.9%) at the same price. Always prefer io2 over io1 for new workloads.

**io2 Block Express** is the highest-performance EBS option: up to 256,000 IOPS and 4,000 MB/s throughput, and sub-millisecond latency. Available on `r5b`, `x2idn`, `x2iedn` instances.

### st1 and sc1 (HDD)

HDD volumes **cannot be used as boot volumes**.

- **st1** (Throughput Optimized): sequential workloads — Kafka, log processing, Hadoop, data warehouses reading large files
- **sc1** (Cold HDD): lowest-cost EBS option, for data accessed less than once a day

> **Exam tip:** If the question mentions throughput-heavy sequential workloads (big data, ETL), st1. If it mentions lowest cost with infrequent access, sc1. For anything IOPS-sensitive or a boot volume, use SSD (gp3/io2).

## EBS Snapshots

Snapshots are point-in-time backups of an EBS volume, stored in S3 (managed by AWS — you don't see the S3 bucket directly).

### How snapshots work
- **Incremental**: the first snapshot copies all data; subsequent snapshots copy only changed blocks
- You can take a snapshot while the volume is attached and in use (though quiescing writes gives a cleaner snapshot)
- To restore: create a new EBS volume from the snapshot — it can be in any AZ in that region, or copied to another region first

### Cross-region and cross-account
- **Copy snapshot to another region**: enables DR and migration
- **Share snapshot with another account**: the recipient can copy it to their own account

### Snapshot features

| Feature | What it does |
|---|---|
| **EBS Snapshot Archive** | Move snapshot to archive tier — 75% cheaper storage; restore takes 24–72 hours |
| **Recycle Bin** | Deleted snapshots go to Recycle Bin for 1–365 days before permanent deletion — protects against accidental deletion |
| **Fast Snapshot Restore (FSR)** | Pre-warms snapshot data so volumes restored from it have full performance immediately (no lazy-loading latency). Higher cost. |

### Encryption and snapshots
- Snapshots of **encrypted volumes** are automatically encrypted
- Volumes created from encrypted snapshots are automatically encrypted
- Snapshots of **unencrypted volumes** are unencrypted

**How to encrypt an existing unencrypted volume:**
```
1. Take a snapshot of the unencrypted volume
2. Copy the snapshot, enabling encryption on the copy
3. Create a new volume from the encrypted snapshot copy
4. Attach the new encrypted volume to the instance
```

## EBS Encryption

EBS encryption uses AES-256 and AWS KMS (either the AWS-managed key `aws/ebs` or a customer-managed KMS key).

When a volume is encrypted:
- **Data at rest** is encrypted on the volume
- **Data in transit** between the volume and the instance is encrypted
- Snapshots are encrypted
- Volumes created from those snapshots are encrypted

Encryption has **minimal performance impact** — it is handled by the Nitro hardware and adds negligible latency.

### Account-level default encryption
You can enable **EBS encryption by default** for an entire AWS account and region. Once enabled, all new EBS volumes and snapshot copies are automatically encrypted with the default KMS key. Existing unencrypted volumes are not retroactively encrypted.

## EC2 Instance Store

Instance Store is **physically attached NVMe storage** on the EC2 host. Unlike EBS, the disk is inside the server that runs your instance — there is no network hop.

### Characteristics

| Property | Value |
|---|---|
| **Performance** | Millions of IOPS, very low latency (sub-millisecond) |
| **Durability** | **Ephemeral** — data is lost on stop, hibernate, or terminate |
| **Cost** | Included in instance price — no additional charge |
| **Size** | Fixed per instance type — cannot be resized |
| **Attachment** | Cannot be detached or reattached |

### When data is lost
- Instance is **stopped** or **terminated**
- Underlying **host hardware fails**

Data survives a **reboot** (the instance stays on the same physical host).

### Use cases
- **Buffers and caches** — Redis-style ephemeral caches that are rebuilt on startup
- **Scratch space** — temporary data for batch jobs, ML training runs
- **High-frequency trading or gaming leaderboards** where latency matters more than durability
- Application-level replication handles durability (e.g., Cassandra replicates across nodes)

Instance types with large Instance Store: `i3`, `i3en`, `i4i` (storage-optimized family), `d3`/`d3en` (dense storage), `h1`.

> **Exam tip:** If the scenario says "temporary", "scratch", "cache", "highest possible IOPS", or "data can be lost" — Instance Store. If the scenario says "persist after termination", "durable", or "backup" — EBS.

## Storage Comparison

| | EBS | Instance Store | EFS |
|---|---|---|---|
| **Type** | Block | Block | File (NFS) |
| **Attachment** | One instance (multi-attach: io1/io2 only) | One instance, fixed | Many instances simultaneously |
| **Durability** | Durable (99.999%) | **Ephemeral** | Durable |
| **Persistence** | Survives stop/terminate | Lost on stop/terminate | Independent of instances |
| **Performance** | Up to 256K IOPS (io2 BE) | Millions of IOPS | Scales automatically |
| **Scope** | AZ | Host | Region (multi-AZ) |
| **Use case** | OS, databases | Cache, scratch | Shared filesystem, CMS, container storage |

## EBS-Backed vs Instance Store-Backed AMIs

| | EBS-Backed | Instance Store-Backed |
|---|---|---|
| **Root volume** | EBS volume | Instance Store |
| **Stop/Start** | Supported | Not supported (terminate only) |
| **Boot time** | Fast (< 1 min typically) | Slower |
| **Persistence** | Root volume persists (configurable) | Root volume lost on termination |
| **Common usage** | Almost all modern AMIs | Legacy |

Modern AMIs are almost always EBS-backed. Instance Store-backed AMIs (also called S3-backed) are legacy and rarely appear in practice.

## Working with EBS via boto3

In [ ]:
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

In [ ]:
# Create a gp3 volume with custom IOPS and throughput
volume = ec2.create_volume(
    AvailabilityZone='us-east-1a',
    VolumeType='gp3',
    Size=100,           # GB
    Iops=6000,          # gp3: configurable independently of size (baseline 3000)
    Throughput=250,     # MB/s (baseline 125)
    Encrypted=True,
    TagSpecifications=[
        {
            'ResourceType': 'volume',
            'Tags': [{'Key': 'Name', 'Value': 'app-data'}]
        }
    ]
)
print(f"Volume ID: {volume['VolumeId']}")
print(f"Type: {volume['VolumeType']}, Size: {volume['Size']} GB")
print(f"IOPS: {volume['Iops']}, Throughput: {volume['Throughput']} MB/s")

In [ ]:
# Take a snapshot, copy to another region with encryption
volume_id = 'vol-0abc123def456'

# 1. Create snapshot
snapshot = ec2.create_snapshot(
    VolumeId=volume_id,
    Description='Pre-deployment backup',
    TagSpecifications=[
        {
            'ResourceType': 'snapshot',
            'Tags': [{'Key': 'Purpose', 'Value': 'backup'}]
        }
    ]
)
snap_id = snapshot['SnapshotId']
print(f"Snapshot: {snap_id}")

# 2. Copy to eu-west-1 with encryption
ec2_eu = boto3.client('ec2', region_name='eu-west-1')
copy = ec2_eu.copy_snapshot(
    SourceRegion='us-east-1',
    SourceSnapshotId=snap_id,
    Description='DR copy — eu-west-1',
    Encrypted=True  # encrypt the copy even if source was unencrypted
)
print(f"EU snapshot: {copy['SnapshotId']}")

In [ ]:
# Modify a live gp2 volume → gp3 (online, no downtime)
# This is the recommended migration path from gp2 to gp3
ec2.modify_volume(
    VolumeId='vol-0abc123def456',
    VolumeType='gp3',
    Iops=3000,
    Throughput=125
)
print("Volume modification initiated — no downtime required")

In [ ]:
# Create a Recycle Bin retention rule for snapshots
rbin = boto3.client('rbin', region_name='us-east-1')

rule = rbin.create_rule(
    RetentionPeriod={
        'RetentionPeriodValue': 30,
        'RetentionPeriodUnit': 'DAYS'
    },
    ResourceType='EBS_SNAPSHOT',
    Description='Retain deleted snapshots for 30 days'
)
print(f"Recycle Bin rule: {rule['Identifier']}")

In [ ]:
# Check instance store availability for a given instance type
response = ec2.describe_instance_types(
    InstanceTypes=['i3.xlarge', 'r5.xlarge', 'm5.xlarge']
)

for it in response['InstanceTypes']:
    itype = it['InstanceType']
    storage = it.get('InstanceStorageInfo', {})
    has_nvme = it.get('InstanceStorageSupported', False)
    if has_nvme:
        total_gb = storage.get('TotalSizeInGB', 0)
        disks = storage.get('Disks', [])
        print(f"{itype}: {total_gb} GB instance store ({len(disks)} disk(s))")
    else:
        print(f"{itype}: no instance store")

## Summary

| Concept | Key Takeaway |
|---|---|
| EBS scope | AZ-scoped — attach only to instances in the same AZ |
| gp3 vs gp2 | gp3 preferred: 3,000 IOPS baseline, IOPS/throughput independent of size, ~20% cheaper |
| io1 vs io2 | Always prefer io2 — same price, 100x more durable |
| io2 Block Express | Up to 256K IOPS — for the largest databases |
| st1 | HDD for sequential throughput workloads (big data, logs) — not bootable |
| sc1 | Cheapest HDD — for infrequent access — not bootable |
| Multi-Attach | io1/io2 only, up to 16 instances in same AZ, requires cluster-aware filesystem |
| Snapshots | Incremental, stored in S3; used for backup, cross-region copy, encryption migration |
| Snapshot Archive | 75% cheaper; 24–72h restore time |
| Recycle Bin | Recover accidentally deleted snapshots within 1–365 days |
| Fast Snapshot Restore | Full performance immediately on restore — no lazy-loading; higher cost |
| Encrypt existing volume | Snapshot → copy with encryption → new volume |
| Instance Store | Physically attached, highest IOPS, ephemeral — lost on stop/terminate |
| Instance Store use case | Caches, scratch space, temp data, or app-level replicated data |
| EBS vs Instance Store | EBS = durable, detachable; Instance Store = fastest, ephemeral |